# R2 — Structural-null comparison (Reviewer B point 2 + Reviewer C point 3)

**B.2 (Reviewer B):** "construct a sparse adjacency matrix constrained by the Allen connectome … sampled in an i.i.d. manner under connectomic constraints."  
**C.3 (Reviewer C):** "a random graph with a matched weight distribution."

Both reviewers ask whether the specific Allen topology is necessary for the paper's claims. We answer with a unified **structural-null experiment**: re-run the whole-brain critical-regime simulation on surrogate connectomes that destroy specific structural features while preserving others, and compare the key metrics against the Allen baseline.

**Surrogates:**
- **S1 — weight-shuffle** (addresses C.3): permute the off-diagonal entries of the 60×60 Allen cortical sub-connectome. Preserves the *exact* weight distribution and density (~99%); destroys all topology.
- **S3 — sparse i.i.d. binarized** (addresses B.2): i.i.d. Bernoulli at brain-realistic density. *Note:* the Allen cortical sub-connectome is itself ~99% dense, so an i.i.d. binary at matched density would be an essentially uniform all-to-all matrix. In the response we use a brain-realistic sparsity (~10–20%) to make this surrogate informative.

**Pipeline:**
1. Generate surrogate matrices (`run_C3B2_make_surrogates.py`).
2. Run whole-brain critical-regime simulations on each surrogate, 16 G values, lean output (`run_C3B2_simulate.py`).
3. Compute per-graph metrics (`run_C3_compute_metrics.py`).
4. Build the comparison figure (`run_C3_make_figure.py`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pickle

# run from revision/round1/
sys.path.insert(0, str(Path().resolve()))          # revision_utils, run_*.py
sys.path.insert(0, str(Path().resolve().parent))   # revision/paths.py
import paths
sys.path.insert(0, str(paths.SRC))                 # original src/

DERIVED = paths.DERIVED
FIG_DIR = paths.FIGURES
DERIVED.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Surrogate matrices and diagnostics

Run once to generate `W_allen_60.npy`, `W_surrogate_S1_shuffle.npy`, `W_surrogate_S3_sparse.npy`.

In [ ]:
import importlib, run_C3B2_make_surrogates as mkmod
importlib.reload(mkmod)
mkmod.main()

In [ ]:
# Visualize the three matrices side by side.
W_allen = np.load(DERIVED / 'W_allen_60.npy')
W_S1 = np.load(DERIVED / 'W_surrogate_S1_shuffle.npy')
W_S3 = np.load(DERIVED / 'W_surrogate_S3_sparse.npy')

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, W, ttl in zip(axes, [W_allen, W_S1, W_S3], ['Allen', 'S1 — weight shuffle', 'S3 — sparse i.i.d.']):
    im = ax.imshow(W, cmap='magma', vmin=0, vmax=W.max())
    ax.set_title(ttl)
    fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02)
fig.tight_layout()
plt.show()

## 2. Run simulations

Each surrogate × 16 G values takes ~2 hours wall time. Run from the shell rather than this notebook so they can finish in the background.

```bash
cd revision/round1
nohup python run_C3B2_simulate.py S1 $MBC_SIM_ROOT/Gscan_surrogate_S1/ > /tmp/sim_S1.log 2>&1 &
```

Output files: `data_G0.pkl` … `data_G15.pkl` (each ~165 MB, contains only `R` and `BOLD_R`).

## 3. Per-graph metrics

For each graph (allen and the surrogates), compute:
- `AC1_local[region, G]` — per-region AC1 of `R`.
- `AC1_GS[G]` — AC1 of the global signal $GS(t) = \frac{1}{N}\sum_n z(R_n(t))$.
- `aval_GS[G]` — global avalanche durations at each G.
- `instr` — in-strength derived from the surrogate's own weight matrix.

In [ ]:
import importlib, run_C3_compute_metrics as cm
importlib.reload(cm)
for graph in ['allen', 'S1']:   # add 'S3' here once that simulation has run
    out = cm.compute_metrics_for_graph(graph)
    out_path = DERIVED / f'C3_metrics_{graph}.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(out, f)
    print(f'Saved {out_path}')

## 4. Comparison figure

2×2 layout overlaying Allen and S1 (and S3 once available):
- **(A)** AC1(GS) vs G — does the global criticality signature emerge with random topology?
- **(B)** Global avalanche P(S) at G\* — does scale-free behaviour survive?
- **(C)** AC1(R_n) vs in-strength at G\* — is the in-strength gradient generic or Allen-specific?
- **(D)** Spearman ρ(in-strength, AC1) vs G — gradient strength as a function of coupling.

In [ ]:
import importlib, run_C3_make_figure as fmod
importlib.reload(fmod)
import sys
sys.argv = ['run_C3_make_figure.py', '--graphs', 'allen,S1']
fmod.main()

In [ ]:
from IPython.display import Image
Image(str(FIG_DIR / 'Fig_C3_comparison.png'))